# AlphaGen US RL Training on Colab

这个 notebook 会在 Colab 中完成以下流程：

1. 挂载 Google Drive。
2. 拉取 AlphaGen 仓库并安装依赖。
3. 用 Qlib 下载或复用缓存的美股日频数据。
4. 按 `train / valid / test` 三段时间切分数据。
5. 运行 AlphaGen 的强化学习训练。
6. 把训练产物和分段评估结果保存回 Google Drive。

默认使用 `SP500` 股票池；如果当前 Qlib 数据目录里没有这个股票池，notebook 会自动从已有股票池里挑选一个可用候选。

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ZZZZkp/alphagen.git'
REPO_BRANCH = 'codex/refresh-runtime-and-smoke-tests'

WORKDIR = Path('/content')
REPO_DIR = WORKDIR / 'alphagen'

DRIVE_ROOT = Path('/content/drive/MyDrive/alphagen_us_rl')
DRIVE_DATA_CACHE = DRIVE_ROOT / 'qlib_data' / 'us_data'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs'
LOCAL_QLIB_DIR = WORKDIR / 'qlib_data' / 'us_data'

MAX_BACKTRACK_DAYS = 100
MAX_FUTURE_DAYS = 30
AUTO_TRAIN_START = '2010-01-01'
VALID_TRADING_DAYS = 252
TEST_TRADING_DAYS = 252

SEGMENTS = None
CALENDAR_INFO = None

SEED = 0
POOL_CAPACITY = 10
TRAINING_STEPS = 20_000
PPO_N_STEPS = 512
BATCH_SIZE = 128
PRINT_EXPR = False

PREFERRED_INSTRUMENTS = ('sp500', 'SP500', 'all', 'ALL')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_URL)
print('Drive root:', DRIVE_ROOT)
print(
    'Segments: auto-computed from calendars/day.txt '
    f'(train_start={AUTO_TRAIN_START}, valid_days={VALID_TRADING_DAYS}, '
    f'test_days={TEST_TRADING_DAYS}, backtrack={MAX_BACKTRACK_DAYS}, '
    f'future={MAX_FUTURE_DAYS})'
)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if REPO_DIR.exists():
    run(['git', 'fetch', '--all', '--tags'], cwd=str(REPO_DIR))
    run(['git', 'checkout', REPO_BRANCH], cwd=str(REPO_DIR))
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=str(REPO_DIR))
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

from importlib.metadata import PackageNotFoundError, version as dist_version
from packaging.requirements import Requirement


def load_requirements(path: Path):
    requirements = []
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        requirements.append(Requirement(line))
    return requirements


def split_requirements(requirements):
    satisfied = []
    missing = []
    for req in requirements:
        try:
            current_version = dist_version(req.name)
        except PackageNotFoundError:
            missing.append(str(req))
            continue
        if req.specifier and not req.specifier.contains(current_version, prereleases=True):
            missing.append(str(req))
        else:
            satisfied.append((req.name, current_version))
    return satisfied, missing


run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = load_requirements(REPO_DIR / 'requirements.txt')
satisfied, missing = split_requirements(requirements)
print(f'Reusing {len(satisfied)} requirements from the current Colab runtime.')
if satisfied:
    print('Reused packages:', ', '.join(f'{name}=={version}' for name, version in satisfied))
if missing:
    print('Installing missing or outdated packages:', ', '.join(missing))
    run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade',
        *missing,
    ])
    print('Selected dependencies installed.')
else:
    print('All requirements are already satisfied by the current runtime.')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import google.protobuf
import sklearn
import torch
from sb3_contrib.ppo_mask import MaskablePPO

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('protobuf:', google.protobuf.__version__)
print('scikit-learn:', sklearn.__version__)
print('Torch:', torch.__version__)
print('MaskablePPO import OK:', MaskablePPO.__name__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
import shutil

import pandas as pd


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)


def copy_tree(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    reset_dir(dst)
    shutil.copytree(src, dst)


def download_qlib_us_data(target_dir: Path):
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        run([
            sys.executable,
            '-m',
            'qlib.cli.data',
            'qlib_data',
            '--target_dir',
            str(target_dir),
            '--region',
            'us',
        ])
    except subprocess.CalledProcessError:
        from qlib.tests.data import GetData

        print('CLI 下载失败，尝试调用 qlib.tests.data.GetData().qlib_data(...)')
        GetData().qlib_data(target_dir=str(target_dir), region='us', exists_skip=True)


def build_segments_from_calendar(calendar_path: Path):
    calendar = pd.read_csv(calendar_path, header=None, names=['date'])
    calendar['date'] = pd.to_datetime(calendar['date'])

    required_days = MAX_BACKTRACK_DAYS + MAX_FUTURE_DAYS + VALID_TRADING_DAYS + TEST_TRADING_DAYS + 1
    if len(calendar) < required_days:
        raise ValueError(
            'Qlib US calendar is too short for the requested split strategy: '
            f'{len(calendar)} rows found, but at least {required_days} are required.'
        )

    usable_dates = pd.Index(
        calendar['date'].iloc[MAX_BACKTRACK_DAYS : len(calendar) - MAX_FUTURE_DAYS]
    )
    train_start_idx = int(usable_dates.searchsorted(pd.Timestamp(AUTO_TRAIN_START)))
    last_train_end_idx = len(usable_dates) - VALID_TRADING_DAYS - TEST_TRADING_DAYS - 1
    if train_start_idx > last_train_end_idx:
        raise ValueError(
            'Not enough usable trading days to build train/valid/test splits from the current '
            'Qlib US dataset. Please reduce VALID_TRADING_DAYS/TEST_TRADING_DAYS or provide '
            'a newer dataset.'
        )

    valid_start_idx = last_train_end_idx + 1
    valid_end_idx = valid_start_idx + VALID_TRADING_DAYS - 1
    test_start_idx = valid_end_idx + 1

    def fmt(ts):
        return pd.Timestamp(ts).strftime('%Y-%m-%d')

    segments = {
        'train': (fmt(usable_dates[train_start_idx]), fmt(usable_dates[last_train_end_idx])),
        'valid': (fmt(usable_dates[valid_start_idx]), fmt(usable_dates[valid_end_idx])),
        'test': (fmt(usable_dates[test_start_idx]), fmt(usable_dates[-1])),
    }
    info = {
        'calendar_start': fmt(calendar['date'].iloc[0]),
        'calendar_end': fmt(calendar['date'].iloc[-1]),
        'usable_start': fmt(usable_dates[0]),
        'usable_end': fmt(usable_dates[-1]),
        'calendar_rows': int(len(calendar)),
        'usable_rows': int(len(usable_dates)),
    }
    return segments, info


expected_calendar = DRIVE_DATA_CACHE / 'calendars' / 'day.txt'
if expected_calendar.exists():
    print('Using cached US Qlib data from Google Drive')
    copy_tree(DRIVE_DATA_CACHE, LOCAL_QLIB_DIR)
else:
    print('Downloading US Qlib data to local Colab storage')
    reset_dir(LOCAL_QLIB_DIR)
    download_qlib_us_data(LOCAL_QLIB_DIR)
    produced_calendar = LOCAL_QLIB_DIR / 'calendars' / 'day.txt'
    if not produced_calendar.exists():
        raise FileNotFoundError(
            'Qlib US data download did not create calendars/day.txt. '
            '请检查当前 pyqlib 版本是否还能访问公开数据源，或改成你自己的 provider_uri。'
        )
    copy_tree(LOCAL_QLIB_DIR, DRIVE_DATA_CACHE)

calendar_path = LOCAL_QLIB_DIR / 'calendars' / 'day.txt'
if not calendar_path.exists():
    raise FileNotFoundError(f'Calendar file not found: {calendar_path}')
SEGMENTS, CALENDAR_INFO = build_segments_from_calendar(calendar_path)

instrument_files = sorted((LOCAL_QLIB_DIR / 'instruments').glob('*.txt'))
available_instruments = [path.stem for path in instrument_files]
print('Available instrument universes:', available_instruments[:20])

SELECTED_INSTRUMENT = next((name for name in PREFERRED_INSTRUMENTS if name in available_instruments), None)
if SELECTED_INSTRUMENT is None:
    raise ValueError(
        f'Could not find a supported US instrument universe in {available_instruments}. '
        '请把 PREFERRED_INSTRUMENTS 改成你的数据目录中真实存在的股票池名称。'
    )

instrument_path = LOCAL_QLIB_DIR / 'instruments' / f'{SELECTED_INSTRUMENT}.txt'
if instrument_path.exists():
    instrument_df = pd.read_csv(
        instrument_path,
        sep='	',
        header=None,
        names=['instrument', 'start', 'end'],
    )
    print(
        'Selected instrument coverage:',
        instrument_df['start'].min(),
        '->',
        instrument_df['end'].max(),
    )

print('Selected instrument universe:', SELECTED_INSTRUMENT)
print('Calendar coverage:', CALENDAR_INFO['calendar_start'], '->', CALENDAR_INFO['calendar_end'])
print('Usable AlphaGen range:', CALENDAR_INFO['usable_start'], '->', CALENDAR_INFO['usable_end'])
print('Auto-selected segments:', SEGMENTS)
print('Local Qlib dir:', LOCAL_QLIB_DIR)
print('Drive Qlib cache:', DRIVE_DATA_CACHE)


In [ ]:
import pandas as pd

from qlib.data import D
from qlib.data.dataset.loader import QlibDataLoader
from alphagen_qlib.stock_data import initialize_qlib

if SEGMENTS is None:
    raise RuntimeError("SEGMENTS 还没生成，请先运行数据下载/切分 cell。")

initialize_qlib(str(LOCAL_QLIB_DIR), region="us")

FEATURES = ["$open", "$close", "$high", "$low", "$volume", "$vwap"]
MAX_BACKTRACK_DAYS = 100
MAX_FUTURE_DAYS = 30

cal = pd.Index(D.calendar())
print("Calendar range:", cal[0].strftime("%Y-%m-%d"), "->", cal[-1].strftime("%Y-%m-%d"))
print("Calendar rows:", len(cal))
print()

def diagnose_range(name: str, start_time: str, end_time: str) -> None:
    print("=" * 100)
    print(f"[{name}] requested:", start_time, "->", end_time)

    start_ts = pd.Timestamp(start_time)
    end_ts = pd.Timestamp(end_time)

    start_idx = int(cal.searchsorted(start_ts))
    end_idx = int(cal.searchsorted(end_ts))
    if end_idx >= len(cal):
        print("end_time 超出 calendar，跳过")
        return
    if cal[end_idx] != end_ts:
        end_idx -= 1

    real_start = cal[start_idx - MAX_BACKTRACK_DAYS]
    real_end = cal[end_idx + MAX_FUTURE_DAYS]
    print(
        f"[{name}] padded load range:",
        real_start.strftime("%Y-%m-%d"),
        "->",
        real_end.strftime("%Y-%m-%d"),
    )

    raw = QlibDataLoader(config=FEATURES).load(
        SELECTED_INSTRUMENT,
        real_start,
        real_end,
    )

    print(f"[{name}] raw shape:", raw.shape)
    print(f"[{name}] raw index names:", raw.index.names)
    print(f"[{name}] raw columns:", list(raw.columns))

    dates = pd.Index(raw.index.get_level_values(0).unique()).sort_values()
    instruments = pd.Index(raw.index.get_level_values(1).unique()).sort_values()

    print(f"[{name}] dates in raw index:", len(dates))
    print(f"[{name}] instruments in raw index:", len(instruments))
    print(f"[{name}] expected date x feature panels:", len(dates) * len(FEATURES))

    stacked = raw.stack()
    actual_pairs = (
        stacked.index.droplevel(1).unique()
        if len(stacked) > 0
        else pd.MultiIndex.from_arrays([[], []], names=["datetime", "feature"])
    )
    expected_pairs = pd.MultiIndex.from_product(
        [dates, FEATURES],
        names=[raw.index.names[0], "feature"],
    )
    missing_pairs = expected_pairs.difference(actual_pairs)

    print(f"[{name}] actual date x feature panels after stack():", len(actual_pairs))
    print(f"[{name}] missing date x feature panels:", len(missing_pairs))

    if len(missing_pairs) > 0:
        preview = [
            (d.strftime("%Y-%m-%d"), f)
            for d, f in list(missing_pairs[:20])
        ]
        print(f"[{name}] first missing pairs:", preview)

        missing_by_feature = (
            pd.Series([feat for _, feat in missing_pairs])
            .value_counts()
            .sort_index()
        )
        print(f"[{name}] missing pair counts by feature:")
        print(missing_by_feature.to_string())
    else:
        print(f"[{name}] no fully-missing date x feature panels found.")

    print()
    print(f"[{name}] feature-level all-NaN date counts:")
    for feat in FEATURES:
        wide = raw[feat].unstack(level=1).reindex(index=dates, columns=instruments)
        all_nan_dates = wide.isna().all(axis=1)
        missing_count = int(all_nan_dates.sum())
        print(f"  {feat}: all-NaN dates = {missing_count}")
        if missing_count > 0:
            sample_dates = [d.strftime("%Y-%m-%d") for d in wide.index[all_nan_dates][:10]]
            print(f"    sample dates: {sample_dates}")

    print()
    print(f"[{name}] feature missing-cell ratios:")
    for feat in FEATURES:
        wide = raw[feat].unstack(level=1).reindex(index=dates, columns=instruments)
        ratio = float(wide.isna().mean().mean())
        print(f"  {feat}: {ratio:.4%}")

    print()
    counts_per_date = raw.groupby(level=0).size()
    expected_rows_per_date = len(instruments)
    abnormal_dates = counts_per_date[counts_per_date != expected_rows_per_date]
    print(f"[{name}] dates with abnormal row count in raw index:", len(abnormal_dates))
    if len(abnormal_dates) > 0:
        print(abnormal_dates.head(20).to_string())

for split_name, (start_time, end_time) in SEGMENTS.items():
    diagnose_range(split_name, start_time, end_time)

whole_start = SEGMENTS["train"][0]
whole_end = SEGMENTS["test"][1]
diagnose_range("combined", whole_start, whole_end)


In [ ]:
import torch

from scripts.rl import latest_run, run_single_experiment, status

if SEGMENTS is None:
    raise RuntimeError('SEGMENTS were not initialized. Please run the data-download cell first.')

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEGMENT_ORDER = ('train', 'valid', 'test')
SEGMENT_TUPLES = tuple(SEGMENTS[name] for name in SEGMENT_ORDER)

print('Training device:', DEVICE)
print('Training segments:', dict(zip(SEGMENT_ORDER, SEGMENT_TUPLES)))

run_single_experiment(
    seed=SEED,
    instruments=SELECTED_INSTRUMENT,
    pool_capacity=POOL_CAPACITY,
    steps=TRAINING_STEPS,
    qlib_data_path=str(LOCAL_QLIB_DIR),
    qlib_region='us',
    device=DEVICE,
    segments=SEGMENT_TUPLES,
    ppo_n_steps=PPO_N_STEPS,
    batch_size=BATCH_SIZE,
    print_expr=PRINT_EXPR,
)

RUN_DIR = Path(latest_run())
print('Latest run dir:', RUN_DIR)
status(str(RUN_DIR))


In [ ]:
import json
import pandas as pd
import shutil

from alphagen.data.expression import Feature, Ref
from alphagen.models.linear_alpha_pool import MseAlphaPool
from alphagen_qlib.calculator import QLibStockDataCalculator
from alphagen_qlib.stock_data import FeatureType, StockData, initialize_qlib
from alphagen_qlib.utils import load_alpha_pool_by_path

checkpoint_paths = sorted(
    RUN_DIR.glob('*_steps_pool.json'),
    key=lambda path: int(path.name.split('_', 1)[0]),
)
if not checkpoint_paths:
    raise FileNotFoundError(f'No *_steps_pool.json checkpoint found under {RUN_DIR}')

final_pool_path = checkpoint_paths[-1]
exprs, weights = load_alpha_pool_by_path(str(final_pool_path))
initialize_qlib(str(LOCAL_QLIB_DIR), region='us')

close = Feature(FeatureType.CLOSE)
target = Ref(close, -20) / close - 1

rows = []
for split_name, (start_time, end_time) in SEGMENTS.items():
    data = StockData(
        instrument=SELECTED_INSTRUMENT,
        start_time=start_time,
        end_time=end_time,
        device=DEVICE,
    )
    calculator = QLibStockDataCalculator(data, target)
    pool = MseAlphaPool(
        capacity=max(POOL_CAPACITY, len(exprs)),
        calculator=calculator,
        ic_lower_bound=None,
        l1_alpha=5e-3,
        device=DEVICE,
    )
    pool.force_load_exprs(exprs, weights=weights)
    ic, rank_ic = pool.test_ensemble(calculator)
    rows.append(
        {
            'split': split_name,
            'start_time': start_time,
            'end_time': end_time,
            'n_days': int(data.n_days),
            'n_stocks': int(data.n_stocks),
            'ic': float(ic),
            'rank_ic': float(rank_ic),
            'pool_size': len(exprs),
            'checkpoint': final_pool_path.name,
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_json_path = RUN_DIR / 'segment_metrics.json'
metrics_csv_path = RUN_DIR / 'segment_metrics.csv'
config_json_path = RUN_DIR / 'colab_config.json'

metrics_json_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
metrics_df.to_csv(metrics_csv_path, index=False)
config_json_path.write_text(
    json.dumps(
        {
            'repo_url': REPO_URL,
            'repo_branch': REPO_BRANCH,
            'seed': SEED,
            'pool_capacity': POOL_CAPACITY,
            'training_steps': TRAINING_STEPS,
            'ppo_n_steps': PPO_N_STEPS,
            'batch_size': BATCH_SIZE,
            'instrument': SELECTED_INSTRUMENT,
            'qlib_region': 'us',
            'qlib_data_path': str(LOCAL_QLIB_DIR),
            'segments': SEGMENTS,
            'calendar_info': CALENDAR_INFO,
            'device': str(DEVICE),
            'final_pool_path': final_pool_path.name,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

drive_run_dir = DRIVE_RUNS_DIR / RUN_DIR.name
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(RUN_DIR, drive_run_dir)

print(metrics_df)
print('Saved run directory to:', drive_run_dir)
print('Saved metrics JSON to:', drive_run_dir / 'segment_metrics.json')
print('Saved metrics CSV to:', drive_run_dir / 'segment_metrics.csv')

## 调参建议

- Colab 首次运行建议先保持 `TRAINING_STEPS = 20_000` 做一轮冒烟，确认数据、显卡和保存路径都正常。
- 如果你打算做更完整的训练，可以把 `POOL_CAPACITY` 和 `TRAINING_STEPS` 一起提高；仓库当前默认配置里，`pool_capacity=10` 对应的完整训练规模是 `200_000` steps。
- 如果公开 Qlib 数据下载通道暂时不可用，可以把你自己的 US Qlib 二进制数据目录先放到 `Google Drive/alphagen_us_rl/qlib_data/us_data`，这个 notebook 会优先复用该缓存。